In [1]:
## import dependencies
import os
import re
import sys
import json
import h5py
import numpy as np
import matplotlib.pyplot as plt
import imageio_ffmpeg

from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML

plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

## Add the path to the DiscEvolution directory
sys.path.append('/Users/ben/Downloads/Planet Formation/Code/DiscEvolution Core Accretion')

## import DiscEvolution modules
from DiscEvolution.constants import *
from DiscEvolution.grid import Grid
from DiscEvolution.star import SimpleStar
from DiscEvolution.eos import IrradiatedEOS, LocallyIsothermalEOS, SimpleDiscEOS
from DiscEvolution.disc import *
from DiscEvolution.viscous_evolution import ViscousEvolution, ViscousEvolutionFV, LBP_Solution, HybridWindModel, TaboneSolution
from DiscEvolution.disc import AccretionDisc
from DiscEvolution.dust import *
from DiscEvolution.dust import PlanetesimalFormation
from DiscEvolution.planet_formation import *
from DiscEvolution.diffusion import TracerDiffusion
from DiscEvolution.opacity import Tazzari2016
from DiscEvolution.chemistry import *

In [2]:
## Method to read data from an HDF5 dataset or group

def read_hdf5_array(obj):
    """
    Read data from either an HDF5 dataset or an HDF5 group.
    """

    ## Normal case (the data are stored as one HDF5 dataset)
    if isinstance(obj, h5py.Dataset):
        return np.array(obj)

    ## Ragged case (the data are stored as a group)
    elif isinstance(obj, h5py.Group):
        values = []

        for key in sorted(obj.keys(), key = int):
            values.append(np.array(obj[key]))

        return values

    ## Raise an error if an unexpected HDF5 object is encountered
    else:
        raise TypeError(f"Unexpected HDF5 object type: {type(obj)}")

## Method to load data from an HDF5 file, together with the config used to produce it

def load_data(data_dir, config_dir):
    """
    Load a disc evolution output file (HDF5 only).

    Parameters:
        data_dir: Path to the HDF5 output file.
        config_dir: Path to the JSON config file used for the run.

    Returns:
        data: Dictionary containing the simulation output.
        config: Dictionary containing the simulation config file.
    """

    data = {}

    ## Load the configuration file

    with open(config_dir, "r") as f:
        config = json.load(f)

    ## Open the HDF5 file in read-only mode
    with h5py.File(data_dir, "r") as f:

        ## -----------------------
        ## General simulation data
        ## -----------------------

        general_keys = [
            "R",
            "t",
            "disk_Mdot_star",
            "disk_Mass",
            "Tc",
            "Sigc"]

        for key in general_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = np.array(f[key])

        ## ---------------
        ## Alpha viscosity
        ## ---------------

        if "alpha_SS" in f.attrs:
            data["alpha_SS"] = float(f.attrs["alpha_SS"])

        elif "alpha_SS" in f:
            data["alpha_SS"] = float(f["alpha_SS"][()])

        if config["winds"]["on"]:
            data["psi"] = config["winds"]["psi_DW"]

        ## -------------
        ## Disk profiles
        ## -------------

        disk_keys = [
            "T",
            "time_snap",
            "Sigma_G",
            "Sigma_dust",
            "Sigma_pebbles",
            "Sigma_planetesimals",
            "Vdrift_grains",
            "Vdrift_pebbles",
            "St_grains",
            "St_pebbles",
            "St_planetesimals",
            "e_planetesimals",
            "i_planetesimals",
            "de2_dt",
            "di2_dt",
            "de2_dt_drag",
            "di2_dt_drag",
            "de2_dt_VS_embryo",
            "di2_dt_VS_embryo",
            "de2_dt_VS_pltsml",
            "di2_dt_VS_pltsml",
            "de2_dt_DF",
            "di2_dt_DF"]

        for key in disk_keys:
            ## Only load the quantity if it exists in the file
            if key in f:
                data[key] = read_hdf5_array(f[key])

        ## -----------
        ## Planet data
        ## -----------

        if config["planets"]["include_planets"]:
            planet_keys = [
                "Mcs",
                "Mes",
                "Rp",
                "disk_Mdot_p",
                "Mdot_planetesimal",
                "Mdot_pebble_core",
                "Mdot_pebble_env",
                "Mdot_migration",
                "Mdot_gas",
                "M_iso_planetesimal",
                "M_iso_pebble",
                "X_cores",
                "X_envs"]

            for key in planet_keys:
                if key in f:
                    data[key] = read_hdf5_array(f[key])

    return data, config

## Plot Baseline Model vs In-situ Planetesimal Accretion Model

In [3]:
## Load data and config file

data_dir = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Output/20260818_1041_psi10_Mdot1.0e-08_M0.1_Rd50.h5"
config_dir = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Config/20260817_balogh2026_SI.json"
data, config = load_data(data_dir, config_dir)

data_dir_PA = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Output/20260818_1341_psi10_Mdot1.0e-08_M0.1_Rd50.h5"
config_dir_PA = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Config/20260817_full_accretion.json"
data_PA, config_PA = load_data(data_dir_PA, config_dir_PA)

In [4]:
## Figure output

config_name = os.path.splitext(os.path.basename(config_dir))[0]
figure_dir = f"/Users/ben/Downloads/Planet Formation/DiscEvolution Animations/"

if not(os.path.exists(figure_dir)):
        os.makedirs(figure_dir)

save_figures = True

In [5]:
## Extract data for plotting

R_imp = config["planets"]["Rp"]
Rp = data["Rp"]
Mcs = data["Mcs"]
Mes = data["Mes"]

R_imp_PA = config_PA["planets"]["Rp"]
Rp_PA = data_PA["Rp"]
Mcs_PA = data_PA["Mcs"]
Mes_PA = data_PA["Mes"]

t = data["t"]
t_PA = data_PA["t"]
t_interval = config["simulation"]["t_interval"]

In [6]:
## Matplotlib colors

n_times = len(t_interval)

color_blues = plt.cm.Blues(np.linspace(1, 0.4, n_times))
color_greys = plt.cm.Greys(np.linspace(1, 0.4, n_times))
color_greens = plt.cm.Greens(np.linspace(1, 0.4, n_times))
color_reds = plt.cm.Reds(np.linspace(1, 0.4, n_times))

## Animation frame delay (in milliseconds)

interval = 500
fps = 2

In [7]:
## Animate planet growth tracks, colored by initial radial distance (one frame per timestep)

n_planets = len(Rp)
n_planets_PA = len(Rp_PA)
n_times = len(t_interval)

def initial_radius(track):
    return track[~np.isnan(track)][0]

R0 = [initial_radius(Rp[i]) for i in range(n_planets)]
R0_PA = [initial_radius(Rp_PA[i]) for i in range(n_planets_PA)]

cmap = plt.get_cmap('plasma_r')

R0_all = R0 + R0_PA
norm = LogNorm(vmin = min(R0_all), vmax = max(R0_all))

y_all = np.concatenate(
    [Mcs[i] + Mes[i] for i in range(n_planets)] +
    [Mcs_PA[i] + Mes_PA[i] for i in range(n_planets_PA)])
y_min, y_max = np.nanmin(y_all[y_all > 0]), np.nanmax(y_all)

fig, ax = plt.subplots(figsize = (8, 6))

sm = ScalarMappable(cmap = cmap, norm = norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax = ax)
cbar.set_label('Initial Distance [AU]')

def update(j):
    ax.clear()

    t_now = t_interval[j] * 1e6

    ## Baseline growth tracks (solid)

    for i in range(n_planets):
        mask = t <= t_now

        x = Rp[i][mask]
        y = (Mcs[i] + Mes[i])[mask]
        color = cmap(norm(R0[i]))

        if len(x) > 1:
            ax.plot(x, y, color = color, linestyle = 'solid', linewidth = 2, zorder = 1)

        if len(x) > 0:
            ax.scatter(x[-1], y[-1], color = 'black', marker = 'o', s = 30, zorder = 3)

    ## Planetesimal accretion growth tracks (dashed)

    for i in range(n_planets_PA):
        mask_PA = t_PA <= t_now

        x_PA = Rp_PA[i][mask_PA]
        y_PA = (Mcs_PA[i] + Mes_PA[i])[mask_PA]
        color = cmap(norm(R0_PA[i]))

        if len(x_PA) > 1:
            ax.plot(x_PA, y_PA, color = color, linestyle = 'dashed', linewidth = 2, zorder = 1)

        if len(x_PA) > 0:
            ax.scatter(x_PA[-1], y_PA[-1], color = 'black', marker = 's', s = 30, zorder = 2)

    ax.set_xlim(1e-1, 1e2 / 2)
    ax.set_ylim(y_min * 0.8, y_max * 1.2)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Radial Distance [AU]')
    ax.set_ylabel(r'Total Mass [M$_\oplus$]')
    ax.set_title(f'Planet Growth Tracks: t = {t_interval[j]} Myr')

    style_handles = [
        Line2D([0], [0], color = 'black', linestyle = 'solid', marker = 'o', label = 'Baseline'),
        Line2D([0], [0], color = 'black', linestyle = 'dashed', marker = 's', label = 'In-situ Accretion')]

    ax.legend(handles = style_handles, loc = 'lower right', fontsize = 8)

update(0)
plt.tight_layout()

ani = FuncAnimation(fig, update, frames = n_times, interval = interval, blit = False)

if save_figures:
    gif_path = os.path.join(figure_dir, "growth_tracks_baseline_vs_insitu.gif")
    ani.save(gif_path, writer = PillowWriter(fps = fps))

    mp4_path = os.path.join(figure_dir, "growth_tracks_baseline_vs_insitu.mp4")
    ani.save(mp4_path, writer = FFMpegWriter(fps = fps))

plt.close(fig)

HTML(ani.to_jshtml())

## Plot Disk Winds (psi = 10) Model vs Viscous Disk (psi = 0.01) Model

In [8]:
## Load data and config file

data_dir_winds = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Output/20260818_1341_psi10_Mdot1.0e-08_M0.1_Rd50.h5"
config_dir_winds = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Config/20260817_full_accretion.json"
data_winds, config_winds = load_data(data_dir_winds, config_dir_winds)

data_dir_viscous = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Output/20260818_2025_psi0.01_Mdot1.0e-08_M0.1_Rd50.h5"
config_dir_viscous = "/Users/ben/Downloads/Planet Formation/DiscEvolution Simulations/Config/20260817_full_accretion_psi0.01.json"
data_viscous, config_viscous = load_data(data_dir_viscous, config_dir_viscous)

In [9]:
## Figure output

config_name = os.path.splitext(os.path.basename(config_dir))[0]
figure_dir = f"/Users/ben/Downloads/Planet Formation/DiscEvolution Animations/"

if not(os.path.exists(figure_dir)):
        os.makedirs(figure_dir)

save_figures = True

In [10]:
## Extract data for plotting

R_imp_winds = config_winds["planets"]["Rp"]
Rp_winds = data_winds["Rp"]
Mcs_winds = data_winds["Mcs"]
Mes_winds = data_winds["Mes"]

R_imp_viscous = config_viscous["planets"]["Rp"]
Rp_viscous = data_viscous["Rp"]
Mcs_viscous = data_viscous["Mcs"]
Mes_viscous = data_viscous["Mes"]

t_winds = data_winds["t"]
t_viscous = data_viscous["t"]
t_interval = config_winds["simulation"]["t_interval"]

In [11]:
## Matplotlib colors

n_times = len(t_interval)

color_blues = plt.cm.Blues(np.linspace(1, 0.4, n_times))
color_greys = plt.cm.Greys(np.linspace(1, 0.4, n_times))
color_greens = plt.cm.Greens(np.linspace(1, 0.4, n_times))
color_reds = plt.cm.Reds(np.linspace(1, 0.4, n_times))

In [12]:
## Animate planet growth tracks, colored by initial radial distance (one frame per timestep)

n_planets_winds = len(Rp_winds)
n_planets_viscous = len(Rp_viscous)
n_times = len(t_interval)

def initial_radius(track):
    return track[~np.isnan(track)][0]

R0_winds = [initial_radius(Rp_winds[i]) for i in range(n_planets_winds)]
R0_viscous = [initial_radius(Rp_viscous[i]) for i in range(n_planets_viscous)]

cmap = plt.get_cmap('plasma_r')

R0_all = R0_winds + R0_viscous
norm = LogNorm(vmin = min(R0_all), vmax = max(R0_all))

y_all = np.concatenate(
    [Mcs_winds[i] + Mes_winds[i] for i in range(n_planets_winds)] +
    [Mcs_viscous[i] + Mes_viscous[i] for i in range(n_planets_viscous)])
y_min, y_max = np.nanmin(y_all[y_all > 0]), np.nanmax(y_all)

fig, ax = plt.subplots(figsize = (8, 6))

sm = ScalarMappable(cmap = cmap, norm = norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax = ax)
cbar.set_label('Initial Distance [AU]')

def update(j):
    ax.clear()

    t_now = t_interval[j] * 1e6

    ## Disk winds model growth tracks (solid)

    for i in range(n_planets_winds):
        mask = t_winds <= t_now

        x = Rp_winds[i][mask]
        y = (Mcs_winds[i] + Mes_winds[i])[mask]
        color = cmap(norm(R0_winds[i]))

        if len(x) > 1:
            ax.plot(x, y, color = color, linestyle = 'solid', linewidth = 2, zorder = 1)

        if len(x) > 0:
            ax.scatter(x[-1], y[-1], color = 'black', marker = 'o', s = 30, zorder = 3)

    ## Viscous disk growth tracks (dashed)

    for i in range(n_planets_viscous):
        mask_viscous = t_viscous <= t_now

        x_viscous = Rp_viscous[i][mask_viscous]
        y_viscous = (Mcs_viscous[i] + Mes_viscous[i])[mask_viscous]
        color = cmap(norm(R0_viscous[i]))

        if len(x_viscous) > 1:
            ax.plot(x_viscous, y_viscous, color = color, linestyle = 'dashed', linewidth = 2, zorder = 1)

        if len(x_viscous) > 0:
            ax.scatter(x_viscous[-1], y_viscous[-1], color = 'black', marker = 's', s = 30, zorder = 2)

    ax.set_xlim(1e-1, 1e2 / 2)
    ax.set_ylim(y_min * 0.8, y_max * 1.2)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Radial Distance [AU]')
    ax.set_ylabel(r'Total Mass [M$_\oplus$]')
    ax.set_title(f'Planet Growth Tracks: t = {t_interval[j]} Myr')

    style_handles = [
        Line2D([0], [0], color = 'black', linestyle = 'solid', marker = 'o', label = 'Disk Winds'),
        Line2D([0], [0], color = 'black', linestyle = 'dashed', marker = 's', label = 'Viscous Disk')]

    ax.legend(handles = style_handles, loc = 'lower right', fontsize = 8)

update(0)
plt.tight_layout()

ani = FuncAnimation(fig, update, frames = n_times, interval = interval, blit = False)

if save_figures:
    gif_path = os.path.join(figure_dir, "growth_tracks_winds_vs_viscous.gif")
    ani.save(gif_path, writer = PillowWriter(fps = fps))

    mp4_path = os.path.join(figure_dir, "growth_tracks_winds_vs_viscous.mp4")
    ani.save(mp4_path, writer = FFMpegWriter(fps = fps))

plt.close(fig)

HTML(ani.to_jshtml())